# KazSAT-2


## FIRST STEP 

### LIBRARY LOADING 

In [144]:
import numpy as np
import pandas as pd
import easygui
from collections import defaultdict
import plotly.express as px
import matplotlib.pyplot as plt
import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.colors as mcolors
import tkinter as tk
from tkinter import filedialog
import os
import copy
import re
from tkinter import messagebox
import seaborn as sns
from plotly import subplots
from sklearn.linear_model import LinearRegression

### DICTIONARY LOADING

In [145]:
def dictionaries():
    global datumn, outOf, txt_plots, plotsHtml,plotsMMA,plotsCount,plotsHeatmap,nameOfTheFile,plotsAll,plotsHtml,fileNameDirectory,filesMerging, txtMerge, plotsOutOf, outPlots, regressionPlots
    filesMerging = defaultdict(list)
    datumn = defaultdict(list)
    txt_plots = defaultdict(list)
    nameOfTheFile = []
    plotsAll = defaultdict(list)
    plotsHtml = defaultdict(list)
    plotsHtml[f'MultiSubPlots'].append(f'MultiPlot.html')
    plotsHtml = defaultdict(list)
    plotsMMA = defaultdict(list)
    plotsCount = defaultdict(list)
    plotsHeatmap = defaultdict(list)
    outOf = defaultdict(list)
    plotsOutOf = defaultdict(list)
    outPlots = defaultdict(list)
    regressionPlots = defaultdict(list)

In [146]:
dictionaries()

### CHOOSING FOLDERS 

In [147]:
def folderCount(event):
    global file_quantityM
    file = chooseFolder.get() ### ПОЛУЧЕНИЕ ТЕКСТА ИЗ TKINTER
    numbers = file.split('Введите количество папок:')[1] ### ВЫДЕЛЕНИЕ ЧИСЛА ИЗ ТЕКСТА
    file_quantityM = int(numbers) ### КОВРЕТИРОВАНИЕ ТЕКСТА В ЧИСЛО

### MERGING TXT FILES 

file_names = easygui.fileopenbox(filetypes="*.txt",multiple=True)

txt = pd.read_csv(file_names[0], sep = '\t',header = 2, encoding = 'ANSI', low_memory = False).drop([0])
txt.rename(columns = {'   Дата         Время       Значение             ':'Дата Время Значение'}, inplace = True)
txt[['Дата', 'Дата и Время','Значение']] = txt['Дата Время Значение'].str.split(expand = True).drop(columns = 3)
txt['Дата и Время'] = [x.split('.')[0] for x in txt['Дата и Время']]
txt['Дата Время Значение'] = [x.split('.')[2] + '-' + x.split('.')[1] + '-' + x.split('.')[0] for x in txt['Дата']]
txt['Дата и Время'] = txt['Дата Время Значение'] + ' ' + txt['Дата и Время']
txt.drop(columns = 'Дата Время Значение', inplace = True)

In [148]:
def Merge():
    global file_quantity, nameOfTheFile
    for i in np.arange(file_quantityM): #КОЛИЧЕСТВО РАЗ, СКОЛЬКО БУДУЬ ОТКРЫВАТЬСЯ ДИАЛОГОВЫЕ ОКНА ДЛЯ ВЫБОРА ПАПКИ
        directory = filedialog.askdirectory() #ОТКРЫТИЕ ПАПКИ (ВЫВЕДЕТ ПУТЬ К ПАПКЕ)
        directory1 = []
        for root, dirs, files in os.walk(directory): #ПОИСК ПУТЕЙ ВСЕХ TXT ФАЛЙЛОВ В ДИРЕКТОРИИ
            for name in files:
               if name.endswith(".txt"):
                   directory1.append(os.path.join(root, name))
        for j in directory1: #ОТКРЫТИЕ TXT ФАЙЛОВ, ИХ СОРТИРОВКА ПО НАЗВАНИЮ ПАРАМЕТРОВ И СОХРАНЕНИЕ В СЛОВАРЬ 
            if 'Merged' in j:
                txt = pd.read_csv(j, sep = '\t', low_memory = False)
                txt['Дата и Время'] = txt['Дата и Время'].astype('datetime64[ns]')
                txt['Дата'] = txt['Дата'].astype('datetime64[ns]')
                txt['Дата'] = pd.to_datetime(txt['Дата'], dayfirst=True).dt.date
                filesMerging[j.split('-')[1]].append(txt)
            else:
                txt = pd.read_csv(j, sep = '\t',header = 2, encoding = 'ANSI', low_memory = False).drop([0])
                txt.rename(columns = {'   Дата         Время       Значение             ':'Дата Время Значение'}, inplace = True)
                txt[['Дата', 'Дата и Время','Значение']] = txt['Дата Время Значение'].str.split(expand = True).drop(columns = 3)
                txt['Дата и Время'] = [x.split('.')[0] for x in txt['Дата и Время']]
                txt['Дата Время Значение'] = [x.split('.')[2] + '-' + x.split('.')[1] + '-' + x.split('.')[0] for x in txt['Дата']]
                txt['Дата и Время'] = txt['Дата Время Значение'] + ' ' + txt['Дата и Время']
                txt.drop(columns = 'Дата Время Значение', inplace = True)
                if len(txt[txt['Значение']=='Ом']) > 0:
                    txt.drop(txt[txt['Значение']=='Ом'].index, inplace = True)
                if len(txt[txt['Значение']=='мВ']) > 0:
                    txt.drop(txt[txt['Значение']=='мВ'].index, inplace = True)
                txt['Значение'] = txt['Значение'].astype('float64')
                txt['Дата'] = pd.to_datetime(txt['Дата'], dayfirst=True).dt.date
                txt['Дата и Время'] = txt['Дата и Время'].astype('datetime64[ns]')
                txt['Дата и Время'] = pd.to_datetime(txt['Дата и Время'], dayfirst = True,format= "%d.%m.%Y %H:%M:%S") 
                appendix = pd.read_csv(j, sep = '\t',encoding = 'ANSI', low_memory = False).head(0).columns[0].split(' ')[2]
                appendix = appendix.replace('/', '_')
                filesMerging[appendix].append(txt) 
    for name in filesMerging.keys():
        name = name.replace('/', '_')
        nameOfTheFile.append(f'{name}')
        txt_plots[name].append(pd.concat(filesMerging[name][:],ignore_index=True))
        txt_plots[name][0].drop_duplicates(subset=['Дата и Время'], inplace = True)
        txt_plots[name][0].sort_values(by =['Дата','Дата и Время'], inplace = True)
    file_quantity = len(nameOfTheFile)

### CHOOSING FILES

In [149]:
def fileSelecting():
    global file_names, file_quantity
    file_names = easygui.fileopenbox(filetypes="*.txt",multiple=True)
    file_quantity = len(file_names)

### READING FILES

In [150]:
def readTxt():
    global nameOfTheFile
    for name,i in zip(file_names,range(len(file_names))):
        if 'Merged' in name:
            txt = pd.read_csv(name, sep = '\t')
            txt['Дата и Время'] = txt['Дата и Время'].astype('datetime64[ns]')
            txt['Дата'] = txt['Дата'].astype('datetime64[ns]')
            txt['Дата'] = pd.to_datetime(txt['Дата'], dayfirst=True).dt.date
            txt['Дата и Время'] = [y + ' ' +x.split(' ')[1] for y in txt['Дата'] for x in txt['Дата и Время']]
            nameOfTheFile.append(name.split('-')[1])
            txt_plots[f'{nameOfTheFile[i]}'].append(txt)
        else:
            appendix = pd.read_csv(name, sep = '\t',encoding = 'ANSI', low_memory = False).head(0).columns[0].split(' ')[2]
            appendix = appendix.replace('/', '_')
            txt = pd.read_csv(name, sep = '\t',header = 2, encoding = 'ANSI', low_memory = False).drop([0])
            txt.rename(columns = {'   Дата         Время       Значение             ':'Дата Время Значение'}, inplace = True)
            txt[['Дата', 'Дата и Время','Значение']] = txt['Дата Время Значение'].str.split(expand = True).drop(columns = 3, axis = 1)
            txt['Дата и Время'] = [x.split('.')[0] for x in txt['Дата и Время']]
            txt['Дата Время Значение'] = [x.split('.')[2] + '-' + x.split('.')[1] + '-' + x.split('.')[0] for x in txt['Дата']]
            txt['Дата и Время'] = txt['Дата Время Значение'] + ' ' + txt['Дата и Время']
            txt.drop(columns = 'Дата Время Значение', axis = 1, inplace = True)
            txt['Значение'] = txt['Значение'].astype('float64')
            txt['Дата'] = pd.to_datetime(txt['Дата'], dayfirst=True).dt.date
            txt['Дата и Время'] = txt['Дата и Время'].astype('datetime64[ns]')
            txt['Дата и Время'] = pd.to_datetime(txt['Дата и Время'], dayfirst = True, format= "%d.%m.%Y %H:%M:%S") 
            nameOfTheFile.append(appendix)
            txt_plots[f'{nameOfTheFile[i]}'].append(txt)

## SECOND STEP 

### CREATING DATASETS

In [151]:
def dataSets(data,i):
        global df, df_Heatmap, df_values, df_sheet4
        ### creaating main dataset
        df = pd.DataFrame(columns = 'Макс Мин Среднее Мода'.split(' '))
        df['Макс'] = data.groupby(by=['Дата'],sort = False )[['Значение']].max()
        df['Мин'] = data.groupby(by=['Дата'],sort = False )[['Значение']].min()
        df['Среднее'] = data.groupby(by=['Дата'],sort = False )[['Значение']].mean().round(2)
        df['Мода'] = data.groupby('Дата',sort = False )['Значение'].apply(lambda x:x.mode()[0])

        ### creaating dataset for values count plot histogram
        df_values = pd.DataFrame(columns = 'Количество'.split(' '))
        df_values['Количество'] = data.groupby(by=['Дата','Значение'], sort = [False])[['Значение']].count()
        df_values_plot_min = df_values.groupby(level = 'Дата').head(1)
        df_values_plot_min = df_values_plot_min.reset_index()
        df_Heatmap = pd.DataFrame()
    
        df_values_plot_min['Дата'] = [x.strftime('%Y-%B') for x in df_values_plot_min['Дата']]
        df_Heatmap['Год'] = [x.strftime('%Y') for x in df.index]
        df_Heatmap['Месяц'] = [x.strftime('%B') for x in df.index]
        df_Heatmap['Макс'] = [x for x in df['Макс']]
        df_Heatmap = df_Heatmap.groupby(by=['Год', "Месяц"])[['Макс']].max().reset_index()

        ### for sheet 3: max/min plots by year and month
        df_sheet3 = pd.DataFrame()
        df_forsheet = pd.DataFrame()
        df_sheet3['Год'] = [x.strftime('%Y') for x in df.index]
        df_sheet3['Месяц'] = [x.strftime('%B') for x in df.index]
        df_sheet3['Макс'] = [x for x in df['Макс']]
        df_sheet3 = df_sheet3.groupby(by = ['Год', 'Месяц'])[['Макс']].max().reset_index()
        df_forsheet['Год'] = [x.strftime('%Y') for x in df.index]
        df_forsheet['Месяц'] = [x.strftime('%B') for x in df.index]
        df_forsheet['Мин'] = [x for x in df['Мин']]
        df_forsheet = df_forsheet.groupby(by = ['Год', 'Месяц'])[['Мин']].min().reset_index()
        df_sheet3['Мин'] = [x for x in df_forsheet['Мин']]
        dates_in_order1 = pd.date_range(start='2022-01-01', end='2022-12-01', freq='MS')
        months_in_order1 = dates_in_order1.map(lambda x: x.month_name()).to_list()
        df_sheet3['Месяц'] = pd.Categorical(df_sheet3['Месяц'], categories=months_in_order1, ordered=True)
        df_sheet3.sort_values(by = ['Год', "Месяц"],  inplace = True,ascending=True)
  
        ### heatmap months order:
        dates_in_order = pd.date_range(start='2022-09-01', end='2023-08-01', freq='MS')
        months_in_order = dates_in_order.map(lambda x: x.month_name()).to_list()
        df_Heatmap['Месяц'] = pd.Categorical(df_Heatmap['Месяц'], categories=months_in_order, ordered=True)
        df_Heatmap.sort_values(by = ['Год', "Месяц"],  inplace = True,ascending=True)

        ### Sheet 4: every year increment 
        df_sheet4 = pd.DataFrame()
        df_sheet4['Год'] = [x for x in df_sheet3['Год']]
        df_sheet4['Макс'] = [x for x in df_sheet3['Макс']]
        df_sheet4 = df_sheet4.groupby('Год')[['Макс']].max().reset_index()
        def increments():
            global increment
            increment = ['Nan']
            for i in range(len(df_sheet4['Макс'])):
                if i <= len(df_sheet4['Макс']) - 2:
                    increment.append(df_sheet4['Макс'][i+1] - df_sheet4['Макс'][i])
        increments()
        df_sheet4['Прирост'] = [x for x in increment]
        
        ### saving datasets to dictionary
        datumn[f'{nameOfTheFile[i]}'].append(df)
        datumn[f'{nameOfTheFile[i]}'].append(df_values)
        datumn[f'{nameOfTheFile[i]}'].append(df_values_plot_min)
        datumn[f'{nameOfTheFile[i]}'].append(df_Heatmap)
        datumn[f'{nameOfTheFile[i]}'].append(df_sheet3)
        datumn[f'{nameOfTheFile[i]}'].append(df_sheet4)

In [152]:
def dataRead():
    for i in np.arange(0,file_quantity):
        dataSets(txt_plots[nameOfTheFile[i]][0],i)

## THIRD STEP

### SETTEING COLORS

In [153]:
colors = ['#2c2ca0','#a02c2c','#2ca02c','#7f0eff',
          '#ff7f0e','#0e8eff','#da0074',
          '#8b4513','#556b2f','#c2185b',
          '#004a19','#00314a', '#7f00cc','#004dcc','#cc004d',]
additional_colors = ['#ff1493', '#32cd32', '#8a2be2', '#ff4500',  
                     '#4682b4', '#b22222', '#228b22', '#9400d3',  
                     '#ffa500', '#20b2aa', '#ff00ff', '#dc143c',  
                     '#00ced1', '#9370db', '#708090', '#6a5acd']
more_colors = ['#1e90ff', '#ff6347', '#2f4f4f', '#8b0000',  
               '#556b2f', '#483d8b', '#008080', '#b8860b',  
               '#4682b4', '#d2691e', '#5f9ea0', '#ff69b4',  
               '#4b0082', '#7cfc00', '#9932cc', '#ff8c00',  
               '#8fbc8f', '#dc143c', '#00ff7f', '#adff2f']
colors = colors + additional_colors + more_colors

### MAKING PLOTS

In [154]:
def allValuesPlot():
    directory = filedialog.askdirectory()
    for i in np.arange(0,file_quantity):
        if len(plotsAll[f'PlotsAll']) < file_quantity: 
            xaxis = txt_plots[f'{nameOfTheFile[i]}'][0]['Дата и Время']
            yaxis = txt_plots[f'{nameOfTheFile[i]}'][0]['Значение']       
            plotsAll[f'PlotsAll'].append(go.Scatter(x = xaxis, y = yaxis,  
                name = f'{nameOfTheFile[i]}',line_shape='hv',mode = 'lines+markers',line=dict(shape='vhv')))
            plotAll = go.Figure(data=plotsAll[f'PlotsAll'][i], layout=dict())
            plotAll.update_layout(title = f'{nameOfTheFile[i]}')
            plotsHtml['PlotsAll'].append(f'{directory}/{nameOfTheFile[i]}.html')
            plotAll.write_html(plotsHtml['PlotsAll'][i])
        else: 
            break

In [155]:
def allValuesMixedPlot():
    directory = filedialog.askdirectory()
    for i in np.arange(0,file_quantity):
        if len(plotsAll[f'PlotsAll']) < file_quantity: 
            xaxis = txt_plots[f'{nameOfTheFile[i]}'][0]['Дата и Время']
            yaxis = txt_plots[f'{nameOfTheFile[i]}'][0]['Значение']       
            plotsAll[f'PlotsAll'].append(go.Scatter(x = xaxis, y = yaxis,  
                name = f'{nameOfTheFile[i]}',line_shape='hv',mode = 'lines+markers',line=dict(shape='vhv')))
        else: 
            break
    multiAll = go.Figure(data=plotsAll[f'PlotsAll'], layout=dict())
    multiAll.write_html(f'{directory}/MultiPlot.html')

In [156]:
def minMaxPlot():
    directory = filedialog.askdirectory()
    r = ['max','min','avg']
    color = ['#a02c2c','#2c2ca0','#2ca02c']
    for p1 in np.arange(0,file_quantity):
        if len(plotsMMA['Max']) < file_quantity: 
            xaxis = datumn[f'{nameOfTheFile[p1]}'][0].index
        ### minimum, maximum and average plots in one 
            for loc in range(0,3): 
                yaxis = datumn[f'{nameOfTheFile[p1]}'][0].iloc[:,loc]
                plotsMMA[f'{nameOfTheFile[p1]}'].append(go.Scatter(x = xaxis, y = yaxis, name = f'{r[loc]}',
                    marker = {'color': f'{color[loc]}'}, line_shape='hv',mode = 'lines + markers'))   
            multiMMA = go.Figure(data=plotsMMA[f'{nameOfTheFile[p1]}'], layout=dict())
            multiMMA.update_layout(title = f'{nameOfTheFile[p1]}')
            plotsHtml['plotsMMA'].append(f'{directory}/{nameOfTheFile[p1]} Max,Min,Avg.html')
            multiMMA.write_html(plotsHtml['plotsMMA'][p1])
            ### maximim and minimum plots in subplot
            yaxis = datumn[f'{nameOfTheFile[p1]}'][0].iloc[:,0]
            plotsMMA['Max'].append(go.Scatter(x = xaxis, y = yaxis, name = f'{nameOfTheFile[p1]}',
                    line_shape='spline',mode = 'lines + markers', marker = {'color': f'{colors[p1]}'},legendgroup=f'group{p1}'))
            yaxis = datumn[f'{nameOfTheFile[p1]}'][0].iloc[:,1]
            plotsMMA['Min'].append(go.Scatter(x = xaxis, y = yaxis, name = f'{nameOfTheFile[p1]}',
                    line_shape='hv',mode = 'lines + markers',marker = {'color': f'{colors[p1]}'},legendgroup=f'group{p1}',showlegend=False))
        else:
            break

In [157]:
def minMaxMixedPlot():
    plotsSub = make_subplots(rows=2, cols=1,row_titles = ['Max','Min'])
    directory = filedialog.askdirectory()
    for p1 in np.arange(0,file_quantity):
        if len(plotsMMA['Max']) < file_quantity: 
            xaxis = datumn[f'{nameOfTheFile[p1]}'][0].index
            yaxis = datumn[f'{nameOfTheFile[p1]}'][0].iloc[:,0]
            plotsMMA['Max'].append(go.Scatter(x = xaxis, y = yaxis, name = f'{nameOfTheFile[p1]}',
                    line_shape='spline',mode = 'lines + markers', marker = {'color': f'{colors[p1]}'},legendgroup=f'group{p1}'))
            yaxis = datumn[f'{nameOfTheFile[p1]}'][0].iloc[:,1]
            plotsMMA['Min'].append(go.Scatter(x = xaxis, y = yaxis, name = f'{nameOfTheFile[p1]}',
                    line_shape='hv',mode = 'lines + markers',marker = {'color': f'{colors[p1]}'},legendgroup=f'group{p1}',showlegend=False))
    for i in plotsMMA['Max']:
        plotsSub.add_trace(i, row = 1, col = 1) 
    for i in plotsMMA['Min']:
        plotsSub.add_trace(i, row = 2, col = 1) 
    plotsHtml[f'plotsSub'].append(f'{directory}/SubPlot.html')
    plotsSub.write_html(plotsHtml[f'plotsSub'][0])

In [158]:
def countPlot():
    directory = filedialog.askdirectory()
    for p2 in np.arange(0,file_quantity):
        if len(plotsCount['Min']) < file_quantity: 
            plotsCount['Min'].append(px.histogram(datumn[f'{nameOfTheFile[p2]}'][2], x = 'Дата', y = 'Количество',color = 'Значение',
                     text_auto=True))
            plotCount = go.Figure(data=plotsCount['Min'][p2], layout=dict())
            plotCount.update_layout(title = f'{nameOfTheFile[p2]}')
            plotsHtml['Min'].append(f'{directory}/{nameOfTheFile[p2]}Histmin.html')
            plotCount.write_html(plotsHtml['Min'][p2])
        else:
            break

In [159]:
def heatmapPlot():
    global directory
    directory = filedialog.askdirectory()
    for p in np.arange(0,file_quantity):
        if len(plotsHeatmap['OnePlot']) < file_quantity: 
            dataHeatmap = datumn[f'{nameOfTheFile[p]}'][3]
            plotsHeatmap['OnePlot'].append(go.Heatmap(x = dataHeatmap['Год'], y = dataHeatmap['Месяц'], z = dataHeatmap['Макс']))
            plotHeatmap = go.Figure(data = plotsHeatmap['OnePlot'][p])
            plotHeatmap.update_layout(title = dict(text = f'{nameOfTheFile[p]}'))
            plotsHtml['HeatMap'].append(f'{directory}/{nameOfTheFile[p]}Heatmap.html')
            plotHeatmap.write_html(plotsHtml['HeatMap'][p])
        else:
            break

In [160]:
def regressionPlot():
    directory = filedialog.askdirectory()
    for p in np.arange(0,file_quantity):
         if len(regressionPlots['Regression']) < file_quantity: 
             reg = pd.DataFrame(columns = 'x y'.split(' '))
             reg['y'] = datumn[nameOfTheFile[p]][0]['Макс']
             for x, y in zip(datumn[nameOfTheFile[p]][0].index, range(len(datumn[nameOfTheFile[p]][0].index))):
                reg.iloc[y,0] = float(x.strftime('%y')) + float(x.strftime('%m'))/12 + float(x.strftime('%d'))/365
             reg['sin'] = np.sin(reg['y']/10)
             time = reg['x'].iloc[-1]
             timedata = pd.DataFrame({'x': time + np.arange(1, 366) / 365})
             predreg = pd.DataFrame(pd.concat([reg['x'], timedata['x']],ignore_index=True), columns = ['x'])
             predreg['x'] = predreg['x'].astype('float64')
             # Генерация синусоидальных данных
             dates = reg['x'].astype('float64')
             datespred = predreg['x'].astype('float64')
             values = reg['y'].astype('float64')
             f = 13
             s = 0
             f1 = f * 1
             f2 = f * 2    
             f3 = f / 2  
             X = np.column_stack([
                 np.sin(f1 * (dates - s)), np.cos(f1 * (dates - s)),
                 np.sin(f2 * (dates - s)), np.cos(f2 * (dates - s)),
                 np.sin(f3 * (dates - s)), np.cos(f3 * (dates - s)),
                 dates 
             ])
             y = values
             model = LinearRegression()
             model.fit(X, y)
             X_pred = np.column_stack([
                 np.sin(f1 * (datespred - s)), np.cos(f1 * (datespred - s)),
                 np.sin(f2 * (datespred - s)), np.cos(f2 * (datespred - s)),
                 np.sin(f3 * (datespred - s)), np.cos(f3 * (datespred - s)),
                 datespred  
             ])
             y_pred = model.predict(X_pred)
             # Визуализация
             fig = go.Figure()
             fig.add_traces([go.Scatter(x = dates, y = values, name = f'{nameOfTheFile[p]}', line_shape='hv',mode = 'lines + markers'), 
                             go.Scatter(x = datespred, y = y_pred, name = f'{nameOfTheFile[p]} Прогноз', line_shape='spline',mode = 'lines + markers')])
             fig.update_layout(title = f'{nameOfTheFile[p]}')
             plotsHtml['Regression'].append(f'{directory}/{nameOfTheFile[p]}Regression.html')
             fig.write_html(plotsHtml['Regression'][p])

In [161]:
def correletion():
    directory = filedialog.askdirectory()
    corr = pd.DataFrame()
    for i in np.arange(0,file_quantity):
        corr[f'{nameOfTheFile[i]}'] = [x for x in datumn[nameOfTheFile[i]][0]['Макс']]
    corPlot = px.imshow(corr.corr(), title = 'Корреляционная матрица')
    plotsHtml['Correlation'].append(f'{directory}/Correlation.html')
    corPlot.write_html(plotsHtml['Correlation'][0])

### SAVING FILES

In [162]:
def save():
    directory = filedialog.askdirectory()
    for name in filesMerging.keys():
        txt_plots[name][0].to_csv(f'{directory}/-{name}-Merged.txt',sep='\t', index=False)

In [163]:
def saveToExcel():
    directory = filedialog.askdirectory()
    for i in np.arange(0,file_quantity):
          with pd.ExcelWriter(f'{directory}/{nameOfTheFile[i]}.xlsx') as writer:
            datumn[f'{nameOfTheFile[i]}'][0].to_excel(writer, sheet_name=f'Statistic info {nameOfTheFile[i]}')
            datumn[f'{nameOfTheFile[i]}'][1].to_excel(writer, sheet_name=f'Values counts {nameOfTheFile[i]}')
            datumn[f'{nameOfTheFile[i]}'][4].to_excel(writer, sheet_name=f'Maximum and minumums {nameOfTheFile[i]}')
            datumn[f'{nameOfTheFile[i]}'][5].to_excel(writer, sheet_name=f'Increment by year {nameOfTheFile[i]}')

In [164]:
def saving():
    answer = messagebox.askquestion("Saving to excel", "хотите сохранить в excel файл?", icon ='question')
    if answer == 'yes':
        saveToExcel()

In [165]:
def savingTxt():
    answer = messagebox.askquestion("Saving to txt", "хотите сохранить в txt?", icon ='question')
    if answer == 'yes':
        save()

## FOURTH STEP

### DIALOG WINDOW


In [166]:
def foldersShow(event):
    global folderMerge, folderAdd
    folderMerge = tk.Button(text="Объединить файлы в папке", command=lambda:[Merge(),dataRead(),saving(),savingTxt()])
    folderMerge.place(x = 5, y = 120)

In [167]:
def folderSelecting():
    global chooseFolder
    chooseFolder = tk.Entry(root, width =30)
    chooseFolder.insert(0,"Введите количество папок: ")
    chooseFolder.place(x = 5, y = 80)
    chooseFolder.bind('<Key>',foldersShow)
    chooseFolder.bind('<Return>',folderCount)

In [168]:
from tkinter import *
root = tk.Tk()    
root.title("analizator K-2")     
root.geometry("707x483")   
  
# Add image file 
bg = PhotoImage(file = 'C:\\Users\\sadykpayev\\Desktop\\Джуниор\\Projects\\analizator\\Kazsat2background.png')
  
# Show image using label 
#label1 = Label(root, image = bg) 
#label1.place(x = 0, y = 0) 

fileLabel = tk.Label(text="Загрузка и обработка файлов", font=("Arial", 10, 'bold'), bg = 'light blue')
fileLabel.place(x = 15, y = 5)

chooseFile = tk.Button(text="Выбрать файлы", command=lambda:[fileSelecting(),readTxt(),dataRead(),saving()])
chooseFile.place(x = 5, y = 40)
choosefolder = tk.Button(text="Выбрать папки", command=folderSelecting)
choosefolder.place(x = 150, y = 40)

dataLabel = tk.Label(text="     Построение графиков      ", font=("Arial", 10, 'bold'), bg = 'light blue')
dataLabel.place(x = 15, y = 160)
allValues = tk.Button(text="Все значения", command=allValuesPlot)
allValues.place(x = 200, y = 240)
allValuesMixed = tk.Button(text="Несколько параметров", command=allValuesMixedPlot)
allValuesMixed.place(x = 15, y = 320)
minmax = tk.Button(text="Максимум, минимум, среднее", command=minMaxPlot)
minmax.place(x = 15, y = 240)
minmaxmixed = tk.Button(text="Несколько параметров: максимум и минимум", command=minMaxMixedPlot)
minmaxmixed.place(x = 15, y = 280)

heat = tk.Button(text="Тепловая карта", command=heatmapPlot)
heat.place(x = 160, y = 200)  

reset = tk.Button(text="Reset", command=dictionaries)
reset.place(x = 660, y = 450)  

regression = tk.Button(text="Прогноз", command=regressionPlot)
regression.place(x = 15, y = 200)

correlation = tk.Button(text="Корреляция", command=correletion)
correlation.place(x = 77, y = 200)

root.resizable(False, False) 
root.mainloop()

In [169]:
txt = pd.read_csv("-ТАПУ1-Merged.txt", sep = '\t')
events = txt[txt['Дата'].isin(['2023-04-28', '2023-04-29','2023-04-30', '2023-05-01', '2023-05-02', 
                              '2023-05-09', '2023-05-10','2023-05-11',
                              '2023-06-15', '2023-06-16','2023-06-17',
                              '2024-03-22', '2024-03-23','2024-03-24']) == True]
events['Дата'] = pd.to_datetime(events['Дата'], errors='coerce').dt.strftime('%Y-%m-%d')
events['Дата и Время'] = pd.to_datetime(events['Дата и Время'], errors='coerce')
events['Дата и Время'] = pd.to_datetime(events['Дата'] + ' ' + events['Дата и Время'].dt.strftime('%H:%M:%S'))